# KCMO Homicide Research Project — Step 0 & Step 1 Pilot Notebook

**Scope:** Kansas City, Missouri (`KCMO`) homicide incidents from 2020 onward (`09A`, `09B`, `09C`), reconstructed from the immutable `Kansas_City_Homicide_Project_Starter.xlsx` original Socrata API JSON records.

This notebook inspects:
1. **Step 0 Foundation:** Reproducible SQLite import (`data/kcmo_homicide.sqlite`), field alias normalization (including 2024's `fire_arm_used_flag`), and the 162-case 2020 research queue.
2. **Step 1 Ten-Case 2020 Pilot:** Multi-agent research + independent verification across 6 workflow strata (`80` audited claims: `58 verified`, `22 unresolved`, `12` verifier corrections/downgrades).

In [1]:
import json
import sqlite3
from src.checks import run_step0_completion_checks, run_step1_pilot_checks
from src.cli import DEFAULT_DB, DEFAULT_EXPORTS_DIR, DEFAULT_QUEUE_CSV, DEFAULT_WORKBOOK

step0_checks = run_step0_completion_checks(DEFAULT_WORKBOOK, DEFAULT_DB, DEFAULT_QUEUE_CSV)
step1_checks = run_step1_pilot_checks(DEFAULT_DB, DEFAULT_EXPORTS_DIR)
print(json.dumps({'step0_checks': step0_checks, 'step1_pilot_checks': step1_checks}, indent=2))

{
  "step0_checks": {
    "workbook_immutable": {
      "passed": true,
      "expected_sha256": "6221d85597ca6b747098278b07816d9a7e70e9acb53548899273d6fe61751717",
      "actual_sha256": "6221d85597ca6b747098278b07816d9a7e70e9acb53548899273d6fe61751717"
    },
    "snapshot_counts": {
      "passed": true,
      "raw_rows_total": 3146,
      "candidate_offense_rows": 3103,
      "excluded_09d_rows": 43,
      "candidate_incidents_total": 974,
      "candidates_by_year": {
        "2020": 162,
        "2021": 124,
        "2022": 158,
        "2023": 166,
        "2024": 138,
        "2025": 121,
        "2026": 105
      },
      "scope_flagged_candidates": 16
    },
    "code_09d_audit_retention": {
      "passed": true,
      "raw_09d_rows_retained": 43,
      "candidates_classified_as_09d": 0,
      "mixed_09a_09d_reports_flagged": 10,
      "mixed_09d_rows_accounted": 23,
      "pure_09d_excluded_rows": 20
    },
    "flags_and_types_integrity": {
      "passed": true,
      "true

In [2]:
conn = sqlite3.connect(DEFAULT_DB)
conn.row_factory = sqlite3.Row

annual_rows = [dict(r) for r in conn.execute('''
    SELECT
        r.dataset_year,
        r.dataset_id,
        COUNT(*) AS total_raw_rows,
        SUM(CASE WHEN r.is_candidate_offense = 1 THEN 1 ELSE 0 END) AS candidate_rows,
        SUM(CASE WHEN r.is_candidate_offense = 0 THEN 1 ELSE 0 END) AS excluded_09d_rows,
        (SELECT COUNT(*) FROM incident_candidates c WHERE c.dataset_year = r.dataset_year) AS candidate_reports,
        (SELECT COUNT(*) FROM incident_candidates c WHERE c.dataset_year = r.dataset_year AND c.scope_review_flag != 'Candidate - verify inclusion') AS scope_flags
    FROM raw_rows r
    GROUP BY r.dataset_year, r.dataset_id
    ORDER BY r.dataset_year
''').fetchall()]

for r in annual_rows:
    print(r)

dataset_year | dataset_id | total_raw_rows | candidate_rows | excluded_09d_rows | candidate_reports | scope_flags
-------------+------------+----------------+----------------+-------------------+-------------------+------------
2020         | vsgj-uufz  | 506            | 498            | 8                 | 162               | 4          
2021         | w795-ffu6  | 381            | 377            | 4                 | 124               | 3          
2022         | x39y-7d3m  | 425            | 425            | 0                 | 158               | 2          
2023         | bfyq-5nh6  | 530            | 526            | 4                 | 166               | 2          
2024         | isbe-v4d8  | 527            | 524            | 3                 | 138               | 1          
2025         | dmnp-9ajg  | 426            | 409            | 17                | 121               | 1          
2026         | f7wj-ckmw  | 351            | 344            | 7                 | 105   

In [3]:
# Inspect Socrata JSON field alias changes across 2020-2026 (including 2024's fire_arm_used_flag)
alias_rows = [dict(r) for r in conn.execute('''
    SELECT
        dataset_year, report_no_field, reported_date_field, reported_time_field,
        zip_code_field, firearm_flag_field, location_format, COUNT(*) AS rows
    FROM raw_rows
    GROUP BY 1, 2, 3, 4, 5, 6, 7
    ORDER BY dataset_year, rows DESC
''').fetchall()]
for r in alias_rows:
    print(r)

dataset_year | report_no_field | reported_date_field | reported_time_field | zip_code_field | firearm_flag_field | location_format            | rows
-------------+-----------------+---------------------+---------------------+----------------+--------------------+----------------------------+-----
2020         | report_no       | reported_date       | reported_time       | zip_code       | firearm_used_flag  | socrata_human_address_dict | 354 
2020         | report_no       | reported_date       | reported_time       | zip_code       | firearm_used_flag  | missing                    | 72  
2020         | report_no       | reported_date       | reported_time       | None           | firearm_used_flag  | socrata_human_address_dict | 64  
2020         | report_no       | reported_date       | reported_time       | None           | firearm_used_flag  | missing                    | 16  
2021         | report_no       | report_date         | report_time         | zip_code       | firearmused_

In [4]:
# Breakdown of the 162 candidate incidents in the 2020 research queue
queue_2020_summary = [dict(r) for r in conn.execute('''
    SELECT
        scope_review_flag,
        source_offense_codes,
        coordinate_precision,
        has_excluded_09d_rows,
        CASE WHEN linked_source_rows = 1 THEN '1 row'
             WHEN linked_source_rows BETWEEN 2 AND 4 THEN '2-4 rows'
             ELSE '5+ rows' END AS row_multiplicity,
        COUNT(*) AS candidate_reports
    FROM incident_candidates
    WHERE dataset_year = 2020
    GROUP BY 1, 2, 3, 4, 5
    ORDER BY candidate_reports DESC
''').fetchall()]
for r in queue_2020_summary:
    print(r)

scope_review_flag            | source_offense_codes | coordinate_precision        | has_excluded_09d_rows | row_multiplicity | candidate_reports
-----------------------------+----------------------+-----------------------------+-----------------------+------------------+------------------
Candidate - verify inclusion | 09A                  | block_or_intersection_point | 0                     | 2-4 rows         | 89               
Candidate - verify inclusion | 09A                  | missing                     | 0                     | 2-4 rows         | 22               
Candidate - verify inclusion | 09A                  | block_or_intersection_point | 0                     | 1 row            | 18               
Candidate - verify inclusion | 09A                  | block_or_intersection_point | 0                     | 5+ rows          | 16               
Candidate - verify inclusion | 09A                  | missing                     | 0                     | 1 row            | 7  

In [5]:
# Step 1 Ten-Case 2020 Pilot Results merged into SQLite
pilot_rows = [dict(r) for r in conn.execute('''
    SELECT
        t.priority_rank AS rank,
        c.report_number,
        t.selection_stratum,
        c.inclusion_decision,
        v.publicly_confirmed_name AS verified_victim,
        c.medical_cause_state,
        c.police_clearance_state,
        c.prosecutor_action_state,
        c.court_outcome_state,
        (SELECT COUNT(*) FROM claims cl WHERE cl.candidate_incident_id = c.candidate_incident_id AND cl.review_state = 'verified') AS verified_claims,
        (SELECT COUNT(*) FROM claims cl WHERE cl.candidate_incident_id = c.candidate_incident_id AND cl.review_state = 'unresolved') AS unresolved_claims
    FROM incident_candidates c
    JOIN research_tasks t ON t.candidate_incident_id = c.candidate_incident_id
    LEFT JOIN deceased_victims v ON v.candidate_incident_id = c.candidate_incident_id AND v.victim_sequence = 1
    WHERE t.priority_rank IS NOT NULL
    ORDER BY t.priority_rank ASC
''').fetchall()]
for r in pilot_rows:
    print(r)
conn.close()

rank | report_number | selection_stratum                             | inclusion_decision                                         | verified_victim                                                       | medical_cause_state | police_clearance_state | prosecutor_action_state | court_outcome_state | verified_claims | unresolved_claims
-----+---------------+-----------------------------------------------+------------------------------------------------------------+-----------------------------------------------------------------------+---------------------+------------------------+-------------------------+---------------------+-----------------+------------------
1    | KC20001088    | A1_ordinary_09A_baseline                      | included_kcmo_homicide                                     | Earl Finch III                                                        | unknown             | verified               | verified                | unresolved          | 9               | 4            